## Pratice for Context Management in OPENAI Agents SDK

In [1]:
from agents import Agent, Runner, function_tool
from agents import RunContextWrapper, TResponseInputItem
from dataclasses import dataclass
import random
import time
import asyncio

In [23]:
@dataclass
class UserProfile:
    id: str
    name: str
    shopping_cart: list[str]


@function_tool
async def get_budget(wrapper: RunContextWrapper[UserProfile]):
    """
    Get the account balance of the user using the user's id and their linked bank account
    """
    print("Getting account balance")
    user_id = wrapper.context.id

    # pretend we are fetching the account balance from a database

    return 100.0


@function_tool
async def search_for_item(wrapper: RunContextWrapper[UserProfile], item: str) -> str:
    """
    Search for an item in the database
    """
    print("Searching for item")
    # randomly generate a price for the item
    price = random.randint(1, 100)
    return f"Found {item} in the database for ${price}.00"


@function_tool
async def get_shopping_cart(wrapper: RunContextWrapper[UserProfile]) -> list[str]:
    print("Getting shopping cart")
    return wrapper.context.shopping_cart


@function_tool
async def add_to_shopping_cart(
    wrapper: RunContextWrapper[UserProfile], items: list[str]
) -> None:
    print("Adding items to shopping cart")
    wrapper.context.shopping_cart.extend(items)


@function_tool
async def purchase_items(wrapper: RunContextWrapper[UserProfile]) -> None:
    print("Purchasing items")

    # we could take the items from the shopping cart and purchase them using some external API
    # for now, we'll just print a message
    print(f"Successfully purchased items: {wrapper.context.shopping_cart}")

In [24]:
shopping_agent = Agent[UserProfile](
    name="Shopping Assistant",
    instructions="You are a shopping assistant dedicated to helping the user with their grocery shopping needs."
    "Your primary role is to assist in creating a shopping plan that fits within the user's budget."
    "Start by getting the user's budget using the tool get_budget."
    "Provide suggestions for items if requested, and always aim to keep the total cost within the user's budget."
    "If the user is nearing or exceeding their budget, inform them and suggest alternatives or adjustments to the shopping list."
    "If the user authorizes it, you can purchase the items using the tool purchase_items.",
    model="gpt-4o-mini",
    tools=[
        get_budget,
        search_for_item,
        get_shopping_cart,
        add_to_shopping_cart,
        purchase_items,
    ],
)

profile1 = UserProfile(id="231", name="John Doe", shopping_cart=[])
print("You are chatting with the Shopping Assistant (type 'exit' to quit) ")

convo_items: list[TResponseInputItem] = []

while True:

    user_input = input("You :")
    if "exit" in user_input:
        break

    convo_items.append({"content": user_input, "role": "user"})

    result = await Runner.run(shopping_agent, convo_items, context=profile1)

    print(f"You : {user_input}")
    print(f"Shopping Assistant : {result.final_output}\n")
    convo_items = result.to_input_list()

You are chatting with the Shopping Assistant (type 'exit' to quit) 
Searching for item
Searching for item
Getting account balance
Adding items to shopping cart
Purchasing items
Successfully purchased items: ['1 liter milk', 'dozen oranges']
You : hi, check if you have 1 ltr milk and a dozen oranges in the database if available check the prices and if in budget add to cart and proceed with the purchase.
Shopping Assistant : I found the following items in the database:

- **1 Liter Milk**: $30.00
- **Dozen Oranges**: $44.00

**Total Cost**: $74.00

Your budget is $100.00, so we are within the budget. I have added the items to your cart and completed the purchase! If you need anything else, feel free to ask.

